# Leakage randomized benchmarking

This notebook demonstrates how to simulate **leakage randomized benchmarking (LRB)** using quax. Leakage occurs when a qubit's population escapes the computational subspace ($|0\rangle$, $|1\rangle$) into higher energy levels (e.g. $|2\rangle$). Standard RB cannot distinguish leakage from ordinary gate errors, so LRB extends the protocol by tracking both the computational-subspace fidelity and the leaked population as a function of circuit depth.

We construct random Clifford sequences, promote them into a qutrit Hilbert space, apply a composite noise model (depolarizing + leakage + seepage), and then fit exponential decay models to extract the leakage and seepage rates.

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

We use `quax` for quantum object manipulation (unitaries, density matrices, superoperators), `jax` for automatic differentiation and JIT compilation, `optax` for gradient-based curve fitting, and `matplotlib` for visualization.

In [ ]:
import quax as qx
import jax
import jax.numpy as jnp

try:
    import optax
except ImportError:
    print("optax not found, optimization will not work. Install optax to enable optimization features.")
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not found, plotting will not work. Install matplotlib to enable plotting features.")

## Parameters

Define the simulation parameters. The **depths** list controls the RB sequence lengths, and **num_randomizations** sets how many random Clifford sequences are sampled at each depth. The three error rates — depolarizing, leakage, and seepage — define the noise model applied after each gate. We will later verify that the fitting procedure recovers these known rates from the simulated data.

In [ ]:
seed = 3847
"""The random seed for reproducibility."""

depths = [1, 3, 7, 13, 27, 53, 81, 121, 161, 201, 401, 801]
"""The depth of the RB circuits."""

num_randomizations = 30
"""The number of random RB circuits to generate for each depth."""

depolarizing_error = 0.03
"""The depolarizing error rate to apply to each gate in the RB circuits."""

leakage_rate = 0.003
"""The leakage rate to apply to each gate in the RB circuits."""

seepage_rate = 0.005
"""The seepage rate to apply to each gate in the RB circuits.""";

## Build the circuits

Construct the RB circuits by randomly sampling elements of the single-qubit Clifford group. For each depth, we draw `depth - 1` random Cliffords, compute their cumulative product, and append the inverse so the ideal sequence compiles to the identity. Shorter sequences are padded with identity gates to a uniform length (`max_depth`) so that all depths can be stacked into a single array and processed in parallel with `vmap`.

In [ ]:
key = jax.random.key(seed)

num_cliffords = qx.ensembles.CLIFFORDS_1Q.ensemble_size[0]
max_depth = max(depths)
depths_arr = jnp.array(depths)


def compute_final_state(random_sequences: qx.Unitary) -> qx.StateVector:
    """
    Compute the final state of an ensemble of random sequences.
    Will reduce on the leading dimension of the ensemble, so the input should be (depth, ..., 2, 2).

    :param random_sequences: (depth, ..., 2, 2) ensemble of random sequences.
    :return: (...) final state vector after applying the sequence.
    """
    ensemble_size = random_sequences.ensemble_size[1:]
    initial_state = qx.zero_state_vector(1, ensemble_size)

    def thunk(state, clifford):
        new_state = clifford @ state
        return new_state, None

    final_state, _ = jax.lax.scan(thunk, initial_state, random_sequences)
    return final_state


def invert_sequence(random_unitaries: qx.Unitary) -> qx.Unitary:
    """
    Given a sequence of 1Q unitaries, compute the overall unitary and invert it.

    :param random_unitaries: Ensemble of random sequences dimension (depth, num_randomizations, 2, 2).
    :return: (depth + 1, num_randomizations, 2, 2) ensemble which accumulate to the identity.
    """
    ensemble_size = random_unitaries.ensemble_size[1:]
    identity = qx.Unitary.from_matrix(
        jnp.broadcast_to(jnp.eye(2, dtype=complex), ensemble_size + (2, 2)),
        ((2,), (2,)),
    )

    def thunk(acc, u):
        return u @ acc, None

    accumulated, _ = jax.lax.scan(thunk, identity, random_unitaries)
    inversion = accumulated.h

    # append the inversion unitary
    random_unitaries = qx.Unitary(
        data=jnp.concatenate([random_unitaries.data, inversion.data[jnp.newaxis]], axis=0),
        num_qubits=1,
    )

    return random_unitaries


# Generate, invert, and pad each depth's Clifford sequence to max_depth
identity_2x2 = jnp.eye(2, dtype=complex)
all_cliffords_data = []
for depth in depths:
    subkey, key = jax.random.split(key)
    rand_ints = jax.random.randint(
        subkey,
        shape=(depth - 1, num_randomizations),
        minval=0,
        maxval=num_cliffords,
    )
    random_cliffords = qx.Unitary(data=qx.ensembles.CLIFFORDS_1Q.data[rand_ints], num_qubits=1)
    random_cliffords = invert_sequence(random_cliffords)

    # Pad shorter sequences to max_depth with identity gates
    pad_length = max_depth - depth
    if pad_length > 0:
        identity_pad = jnp.broadcast_to(identity_2x2, (pad_length, num_randomizations, 2, 2))
        padded = jnp.concatenate([random_cliffords.data, identity_pad], axis=0)
    else:
        padded = random_cliffords.data
    all_cliffords_data.append(padded)

# Stack: (num_depths, max_depth, num_randomizations, 2, 2)
all_cliffords = qx.Unitary(data=jnp.stack(all_cliffords_data), num_qubits=1)

# Mask: True where real gates exist (positions 0..depth-1 for each depth)
gate_mask = jnp.arange(max_depth)[None, :] < depths_arr[:, None]

## Check the final state of each random sequence

As a sanity check, we apply each noiseless Clifford sequence to the $|0\rangle$ state and verify that the probability of measuring $|0\rangle$ is 1.0 at every depth. This confirms that the inversion gate correctly compiles each sequence back to the identity.

In [ ]:
# vmap compute_final_state over the depths dimension (no loop)
all_final_states = jax.vmap(compute_final_state)(all_cliffords)
all_zero_probs = jax.vmap(lambda s: qx.bitstring_probability(s, jnp.array([0])))(all_final_states)

for i, depth in enumerate(depths):
    print(f"Probability of measuring '0' at depth {depth}: {all_zero_probs[i].mean():.4f}")

## Promote to qutrit space

To model leakage we need a third energy level $|2\rangle$. We use `qx.promote` to embed the $2 \times 2$ Clifford unitaries into $3 \times 3$ qutrit unitaries, where the promoted gate acts as identity on the $|2\rangle$ subspace.

In [ ]:
# Promote all cliffords to qutrit space
all_cliffords_3 = qx.promote(all_cliffords, (3,))

## Build the noise model and simulate

We construct a composite noise channel applied after every gate: first **depolarizing** noise (restricted to the qubit subspace), then **leakage** ($|0\rangle, |1\rangle \to |2\rangle$), then **seepage** ($|2\rangle \to |1\rangle$). Each channel is represented as a superoperator, and the full noise channel is their composition. The noisy gate superoperators are applied sequentially to an initial $|0\rangle\langle 0|$ density matrix using `jax.lax.scan`, and all depths and randomizations are processed in parallel via `jax.vmap` and `jax.jit`.

In [ ]:
def compute_final_matrix(random_sequences: qx.SuperOp) -> qx.DensityMatrix:
    """
    Scan-reduce a sequence of SuperOps applied to |0><0|.

    :param random_sequences: (depth, ..., 3, 3, 3, 3) ensemble of SuperOps.
    :return: (...) final density matrix.
    """
    ensemble_size = random_sequences.ensemble_size[1:]
    initial_state = qx.zero_state_matrix(dims=(3,), ensemble_size=ensemble_size)

    def thunk(state, channel):
        new_state = channel @ state
        return new_state, None

    final_state, _ = jax.lax.scan(thunk, initial_state, random_sequences)
    return final_state


def compute_all_probabilities(random_sequences: qx.SuperOp):
    """Compute all qutrit probabilities from a noisy channel sequence."""
    final_dm = compute_final_matrix(random_sequences)
    return qx.probabilities(final_dm)  # (..., 3) -> [P(|0>), P(|1>), P(|2>)]


def probability_ensemble(depolarizing_error, leakage_rate, seepage_rate, all_cliffords_3, gate_mask):
    """
    Build noise channels, apply to promoted cliffords, and compute all probabilities.

    :param depolarizing_error: Depolarizing error rate (scalar).
    :param leakage_rate: Leakage rate (scalar).
    :param seepage_rate: Seepage rate (scalar).
    :param all_cliffords_3: Promoted Clifford unitaries (num_depths, max_depth, num_randomizations, 3, 3).
    :param gate_mask: Boolean mask (num_depths, max_depth) — True for real gates.
    :return: all_probs of shape (num_depths, num_randomizations, 3).
    """
    # Build qubit-subspace depolarizing channel embedded in qutrit space.
    # The d=2 depol superop is placed at the qubit subblock indices [0,1,3,4]
    # of the 9×9 qutrit superop; coherences with |2⟩ are damped by (1-p)
    # and ρ_{22} is left unchanged (identity).
    d_q = 2
    vec_I2 = jnp.ravel(jnp.eye(d_q, dtype=complex))
    S2 = (1 - depolarizing_error) * jnp.eye(d_q**2, dtype=complex) + depolarizing_error * jnp.outer(
        vec_I2, vec_I2.conj()
    ) / d_q
    qubit_idx = jnp.array([0, 1, 3, 4])
    coh_idx = jnp.array([2, 5, 6, 7])
    S9 = jnp.zeros((9, 9), dtype=complex)
    S9 = S9.at[jnp.ix_(qubit_idx, qubit_idx)].set(S2)
    S9 = S9.at[coh_idx, coh_idx].set(1 - depolarizing_error)
    S9 = S9.at[8, 8].set(1.0)
    depol_super = qx.SuperOp.from_matrix(S9, ((3,), (3,)))

    leak_super = qx.kraus_to_superop(qx.leakage_operators(leakage_rate))
    seep_super = qx.kraus_to_superop(qx.seepage_operators(seepage_rate))
    noise_super = qx.compose_superop(seep_super, qx.compose_superop(leak_super, depol_super))

    # Compose noise with all gate SuperOps at once (broadcasting)
    all_noisy = qx.compose_superop(noise_super, qx.unitary_to_superop(all_cliffords_3))

    # Replace padded positions with identity SuperOp
    identity_super_data = jnp.eye(9, dtype=complex).reshape(3, 3, 3, 3)
    mask_nd = gate_mask[..., None, None, None, None, None]
    all_channels_data = jnp.where(mask_nd, all_noisy.data, identity_super_data)
    all_channels = qx.SuperOp(data=all_channels_data, num_qubits=1)

    # vmap compute_all_probabilities over the depths dimension
    return jax.vmap(compute_all_probabilities)(all_channels)


# JIT-compile and run — returns (num_depths, num_randomizations, 3)
all_probs = jax.jit(probability_ensemble)(depolarizing_error, leakage_rate, seepage_rate, all_cliffords_3, gate_mask)
all_zero_probs = all_probs[..., 0]
all_leaked_probs = all_probs[..., 2]

# ⟨Z⟩ = P(|0⟩) - P(|1⟩) = 2·P(|0⟩) + P(|2⟩) - 1
all_exp_z = 2 * all_zero_probs + all_leaked_probs - 1
average_exp_z = jnp.mean(all_exp_z, axis=1)
average_leaked = jnp.mean(all_leaked_probs, axis=1)

for i, depth in enumerate(depths):
    print(f"Depth {depth:3d}: ⟨Z⟩ = {average_exp_z[i]:.4f}, leaked population = {average_leaked[i]:.6f}")

## Plot the state fidelity and the leaked population vs depth

Visualize the two key observables: $\langle Z \rangle$ (a proxy for computational-subspace fidelity) should decay exponentially, while the leaked population $P(|2\rangle)$ should grow and saturate at the steady-state value $L_\infty = \gamma_L / (\gamma_L + \gamma_S)$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(depths, average_exp_z, "o-")
ax1.set_xlabel("Sequence depth")
ax1.set_ylabel("$\\langle Z \\rangle$")
ax1.set_title("$\\langle Z \\rangle$ vs depth")
ax1.set_ylim(-0.05, 1.05)

ax2.plot(depths, average_leaked, "s-", color="tab:red")
ax2.set_xlabel("Sequence depth")
ax2.set_ylabel("Leaked population $P(|2\\rangle)$")
ax2.set_title("Leaked population vs depth")

fig.tight_layout()
plt.show()

## Fit exponential decay models and extract noise parameters

We fit the model $f(m) = A \cdot p^m + B$ to both the $\langle Z \rangle$ and leaked-population curves using L-BFGS optimization (via `optax`). From the leakage fit, the decay rate $q$ and asymptote $B$ directly yield the effective leakage and seepage rates:

$$q = 1 - (\gamma_L + \gamma_S), \qquad B = \frac{\gamma_L}{\gamma_L + \gamma_S}$$

Since our depolarizing channel is restricted to the qubit subspace, the extracted rates should match the nominal values used in the simulation.

In [ ]:
exp_z = jnp.array(average_exp_z)
leaked = jnp.array(average_leaked)


# Model: f(m) = A * p^m + B
def exp_model(params, m):
    A, p, B = params
    return A * p**m + B


def mse_loss(params, m, y):
    return jnp.mean((exp_model(params, m) - y) ** 2)


def fit_exponential(m, y, init_params, num_steps=200):
    """Fit A * p^m + B to data (m, y) using L-BFGS."""
    params = jnp.array(init_params)
    solver = optax.lbfgs()
    opt_state = solver.init(params)
    value_and_grad_fn = jax.value_and_grad(mse_loss)

    @jax.jit
    def step(params, opt_state):
        value, grad = value_and_grad_fn(params, m, y)
        updates, opt_state_new = solver.update(
            grad,
            opt_state,
            params,
            value=value,
            grad=grad,
            value_fn=lambda p: mse_loss(p, m, y),
        )
        params_new = optax.apply_updates(params, updates)
        return params_new, opt_state_new

    for _ in range(num_steps):
        params, opt_state = step(params, opt_state)

    return params


m_jnp = jnp.array(depths, dtype=float)

# --- Fit ⟨Z⟩ decay ---
popt_fid = fit_exponential(m_jnp, exp_z, [1.0, 0.97, -0.1])
A_f, p_f, B_f = popt_fid

# --- Fit leakage growth ---
popt_leak = fit_exponential(m_jnp, leaked, [-0.3, 0.99, 0.35])
A_l, q_l, B_l = popt_leak

### Extract the parameters

The leaked population follows $P(|2\rangle) = A_l \cdot q^m + B_l$, where $q$ is the decay rate and $B_l$ is the steady-state leakage fraction. From these two fitted quantities we can recover the physical rates:

- **Total rate:** $\gamma_L + \gamma_S = 1 - q$
- **Steady-state leakage:** $L_\infty = B_l = \gamma_L / (\gamma_L + \gamma_S)$
- **Leakage rate:** $\gamma_L = L_\infty \cdot (1 - q)$
- **Seepage rate:** $\gamma_S = (1 - L_\infty) \cdot (1 - q)$

This decomposition works because the leakage dynamics form a two-state Markov chain between the computational subspace and $|2\rangle$, whose eigenvalue is $q$ and whose stationary distribution is set by the ratio of the two rates.

In [ ]:
total_rate_extracted = 1 - q_l
L_inf = B_l
leakage_extracted = L_inf * total_rate_extracted
seepage_extracted = (1 - L_inf) * total_rate_extracted

leakage_effective_expected = leakage_rate
seepage_effective_expected = seepage_rate

print(f"\n{'=' * 65}")
print(f"{'Parameter':<30} {'Extracted':>12} {'Known':>12} {'Rel Err':>10}")
print(f"{'-' * 65}")
print(
    f"{'Leakage rate (eff.)':<30} {100 * leakage_extracted:>11.2f}%"
    f" {100 * leakage_effective_expected:>11.2f}% {abs(leakage_extracted - leakage_effective_expected) / leakage_effective_expected:>9.1%}"
)
print(
    f"{'Seepage rate (eff.)':<30} {100 * seepage_extracted:>11.2f}%"
    f" {100 * seepage_effective_expected:>11.2f}% {abs(seepage_extracted - seepage_effective_expected) / seepage_effective_expected:>9.1%}"
)
L_inf_expected = leakage_effective_expected / (leakage_effective_expected + seepage_effective_expected)
print(
    f"{'Steady-state leakage':<30} {100 * L_inf:>11.2f}%"
    f" {100 * L_inf_expected:>11.2f}% {abs(L_inf - L_inf_expected) / L_inf_expected:>9.1%}"
)
print(f"{'-' * 65}")
print(f"{'Leakage rate (nominal)':<30} {100 * leakage_extracted:>11.2f}% {100 * leakage_rate:>11.2f}%")
print(f"{'Seepage rate (nominal)':<30} {100 * seepage_extracted:>11.2f}% {100 * seepage_rate:>11.2f}%")
print(f"{'=' * 65}")


# --- Plot fits overlaid on data ---
m_fine = jnp.linspace(0, max(depths), 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(depths, exp_z, "o", label="Data")
ax1.plot(m_fine, exp_model(popt_fid, m_fine), "-", label=f"Fit: p={p_f:.4f}")
ax1.set_xlabel("Sequence depth")
ax1.set_ylabel("$\\langle Z \\rangle$")
ax1.set_title("$\\langle Z \\rangle$ vs depth")
ax1.set_ylim(-0.05, 1.05)
ax1.legend()

ax2.plot(depths, leaked, "s", color="tab:red", label="Data")
ax2.plot(m_fine, exp_model(popt_leak, m_fine), "-", color="tab:red", label=f"Fit: q={q_l:.4f}")
ax2.set_xlabel("Sequence depth")
ax2.set_ylabel("Leaked population $P(|2\\rangle)$")
ax2.set_title("Leaked population vs depth")
ax2.legend()

fig.tight_layout()
plt.show()